# Tutorial 12 — Direct Preference Optimization

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part IV — Alignment**  
**Follows:** Tutorial 11 (Supervised Fine-Tuning & LoRA)  
**Precedes:** Tutorial 13 (Reward Model Training)

---

## What This Tutorial Covers

RLHF (Reinforcement Learning from Human Feedback) is the alignment technique
behind ChatGPT, Claude, and Gemini. The standard RLHF pipeline has three
stages: SFT → reward model training → PPO. PPO is complex, unstable, and
sensitive to hyperparameters in ways that are hard to debug.

DPO (Rafailov et al., 2023) collapses the reward model and PPO into a
single supervised objective. You train directly on preference data —
pairs of (chosen, rejected) responses — without ever explicitly training
a reward model or running a policy gradient algorithm.

This tutorial covers:

1. **The RLHF objective** — what PPO is actually optimizing, the
   KL-regularized reward maximization problem.
2. **The DPO derivation** — how DPO emerges from the optimal policy of
   the RLHF objective. The math is not optional here; it is the reason
   DPO works.
3. **The DPO loss** — implementation, the reference model, the
   log-probability computation.
4. **Preference data format** — the `(prompt, chosen, rejected)` triple.
5. **The `DPODataset`** — loading, formatting, the forward pass.
6. **The DPO training loop** — two-model forward pass, the `β` parameter.
7. **Monitoring DPO training** — what healthy training looks like,
   the reward margin diagnostic.

---

## 1. The RLHF Objective

The goal of RLHF is to find a policy (language model) $\pi_\theta$ that
maximizes expected reward while not diverging too far from the SFT model
$\pi_{\text{ref}}$:

$$\max_{\pi_\theta} \; \mathbb{E}_{x \sim \mathcal{D},\, y \sim \pi_\theta(\cdot \mid x)}
\left[ r(x, y) \right] - \beta \, D_{\text{KL}}\!\left(\pi_\theta(\cdot \mid x) \,\|\, \pi_{\text{ref}}(\cdot \mid x)\right)$$

where:
- $r(x, y)$ is a reward model score for response $y$ to prompt $x$
- $\beta > 0$ controls how much we penalize divergence from $\pi_{\text{ref}}$
- $D_{\text{KL}}$ is the KL divergence

The KL term is essential. Without it, the policy would optimize the reward
model arbitrarily — eventually generating text that scores highly on $r$
but is nonsense (reward hacking). The KL penalty keeps the policy close to
the SFT model, which we know generates coherent text.

PPO optimizes this objective by alternating between:
1. Sampling responses from $\pi_\theta$ and scoring with $r$
2. Running a clipped policy gradient update on $\pi_\theta$

This requires the reward model, the policy model, a value model, and a
reference model all in memory simultaneously — typically 4× the memory
of a single model.

---

## 2. The DPO Derivation

The key insight is that the KL-regularized objective has a **closed-form
optimal policy**. Setting the functional derivative to zero:

$$\pi^*(y \mid x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y \mid x) \exp\!\left(\frac{1}{\beta} r(x, y)\right)$$

where $Z(x) = \sum_y \pi_{\text{ref}}(y \mid x) \exp(r(x, y) / \beta)$
is the partition function (normalizer).

Rearranging to express the reward in terms of the optimal policy:

$$r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$$

Now substitute this expression into the Bradley-Terry preference model,
which says that the probability of preferring response $y_w$ over $y_l$ is:

$$P(y_w \succ y_l \mid x) = \sigma\!\left(r(x, y_w) - r(x, y_l)\right)$$

[The $\beta \log Z(x)$ terms cancel (since both responses have the same
prompt $x$)]{.mark}:

$$P(y_w \succ y_l \mid x) = \sigma\!\left(\beta \log \frac{\pi^*(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi^*(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)$$

The DPO training objective is the negative log-likelihood of the observed
preferences under this model, with $\pi_\theta$ in place of $\pi^*$:

$$\mathcal{L}_{\text{DPO}}(\pi_\theta) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\left[\log \sigma\!\left(\beta \left(\log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right)\right]$$

**What this means intuitively:**

[Define the **implicit reward**]{.underline}[^implicit_reward] of a response $y$ given prompt $x$ as:

[^implicit_reward]: The implicit reward $\hat{r}(x,y) = \beta \log(\pi_\theta(y|x)/\pi_{\text{ref}}(y|x))$ is "implicit" because it is never explicitly computed as a separate model — it emerges from the ratio of the trained policy to the reference. The reference model serves as a baseline: a response scores high only if the trained policy assigns it *relatively* higher probability than the reference would.

$$\hat{r}(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$$

DPO maximizes the probability that the chosen response has higher implicit
reward than the rejected response. The reference model $\pi_{\text{ref}}$
plays the role of a baseline — the implicit reward is the log-ratio of
how much more (or less) the trained model prefers $y$ compared to the
reference.

[Crucially: **no reward model is needed**.]{.mark} The reward is implicit in the
ratio of the policy to the reference. The partition function $Z(x)$ cancels
because both $y_w$ and $y_l$ share the same prompt.

---

## 3. Computing Log-Probabilities

The DPO loss requires the log-probability of a complete response under
a language model. For an autoregressive model:

$$\log \pi(y \mid x) = \sum_{t=1}^{|y|} \log \pi(y_t \mid x, y_{<t})$$

In practice, we run the full sequence $(x, y)$ through the model and sum
the log-probabilities of the response tokens, masking the prompt tokens
(exactly the same loss mask as SFT):

In [ ]:
import torch
import torch.nn.functional as F

def sequence_log_probs(
    model,
    input_ids:  torch.Tensor,   # (B, T) full sequence: prompt + response
    labels:     torch.Tensor,   # (B, T) response tokens; -100 at prompt positions
) -> torch.Tensor:
    """
    Compute the sum of log-probabilities for each response in the batch.

    Returns:
        log_probs: (B,) — sum of log p(token) for each response token.
    """
    with torch.no_grad() if not model.training else torch.enable_grad():
        logits, _ = model(input_ids)   # (B, T, V)

    # Shift: logits[t] predicts token[t+1]
    # We want log p(labels[t]) = log softmax(logits[t-1])[labels[t]]
    logits_shifted = logits[:, :-1, :]      # (B, T-1, V)
    labels_shifted = labels[:, 1:]          # (B, T-1)

    # Mask prompt positions
    response_mask  = (labels_shifted != -100).float()   # (B, T-1)

    # Replace -100 with 0 for indexing (those positions will be masked out)
    labels_clipped = labels_shifted.clone()
    labels_clipped[labels_shifted == -100] = 0

    # log p for each token position
    log_probs_all = F.log_softmax(logits_shifted, dim=-1)  # (B, T-1, V)
    # Gather log p for the actual next token
    token_log_probs = log_probs_all.gather(
        dim=2,
        index=labels_clipped.unsqueeze(2)
    ).squeeze(2)   # (B, T-1)

    # Sum over response tokens only
    return (token_log_probs * response_mask).sum(dim=1)   # (B,)

---

## 4. The DPO Loss

In [ ]:
def dpo_loss(
    policy_chosen_logps:    torch.Tensor,   # (B,)
    policy_rejected_logps:  torch.Tensor,   # (B,)
    ref_chosen_logps:       torch.Tensor,   # (B,)
    ref_rejected_logps:     torch.Tensor,   # (B,)
    beta:                   float = 0.1,
) -> tuple[torch.Tensor, dict]:
    """
    DPO loss.

    Args:
        policy_*_logps:  sum log-probs from the policy (trained) model
        ref_*_logps:     sum log-probs from the reference (frozen) model
        beta:            KL penalty coefficient

    Returns:
        loss:    scalar loss
        metrics: dict of diagnostic metrics
    """
    # Implicit reward for each response under the policy vs reference
    chosen_reward   = beta * (policy_chosen_logps   - ref_chosen_logps)
    rejected_reward = beta * (policy_rejected_logps - ref_rejected_logps)

    # Reward margin: positive = policy prefers chosen over rejected
    reward_margin   = chosen_reward - rejected_reward

    # DPO loss: negative log-sigmoid of the reward margin
    loss = -F.logsigmoid(reward_margin).mean()

    # Diagnostic metrics
    metrics = {
        'loss':              loss.item(),
        'reward_margin':     reward_margin.mean().item(),
        'chosen_reward':     chosen_reward.mean().item(),
        'rejected_reward':   rejected_reward.mean().item(),
        'accuracy':          (reward_margin > 0).float().mean().item(),
    }
    return loss, metrics

**The `β` parameter** controls the KL penalty. Higher β keeps the policy
closer to the reference model (more conservative updates). Lower β allows
larger divergence (more aggressive optimization of preferences).

- [`β = 0.1` — standard starting point, used in the original DPO paper]{.underline}
- `β = 0.01` — aggressive; use when preference data is high quality
- `β = 0.5` — conservative; use when the reference model is already good
  and you want subtle preference tuning

---

## 5. The DPO Dataset

In [ ]:
from torch.utils.data import Dataset
import json

class DPODataset(Dataset):
    """
    Loads preference data from a JSONL file where each line is:
    {
        "prompt":   "What is the capital of France?",
        "chosen":   "The capital of France is Paris.",
        "rejected": "France doesn't have a capital."
    }

    Returns four tensors per sample:
    - chosen_ids, chosen_labels   (prompt + chosen response)
    - rejected_ids, rejected_labels  (prompt + rejected response)
    """

    def __init__(
        self,
        data_path:  str,
        tokenizer,
        max_length: int = 512,
    ):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.samples    = []

        with open(data_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    self.samples.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

        print(f"DPODataset: {len(self.samples)} preference pairs")

    def _encode_sample(self, prompt: str, response: str):
        """
        Encode (prompt + response) into (input_ids, labels).
        Labels are -100 at prompt positions, token_id at response positions.
        """
        prompt_text   = (f"{SPECIAL_TOKENS['user']}\n{prompt.strip()}\n"
                         f"{SPECIAL_TOKENS['end']}\n"
                         f"{SPECIAL_TOKENS['assistant']}\n")
        response_text = f"{response.strip()}{SPECIAL_TOKENS['end']}\n"

        prompt_ids   = self.tokenizer.encode(prompt_text)
        response_ids = self.tokenizer.encode(response_text)

        # Truncate if needed (preserve prompt, truncate response)
        max_resp = self.max_length - len(prompt_ids) - 1
        response_ids = response_ids[:max_resp]

        input_ids = prompt_ids + response_ids
        labels    = ([-100] * len(prompt_ids)) + response_ids

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(labels,    dtype=torch.long),
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        chosen_ids,   chosen_labels   = self._encode_sample(s['prompt'], s['chosen'])
        rejected_ids, rejected_labels = self._encode_sample(s['prompt'], s['rejected'])
        return chosen_ids, chosen_labels, rejected_ids, rejected_labels


def collate_dpo(batch):
    """Pad all four tensors to the longest sequence in the batch."""
    chosen_ids, chosen_labels, rejected_ids, rejected_labels = zip(*batch)

    def pad(tensors, pad_value):
        max_len = max(t.size(0) for t in tensors)
        out = torch.full((len(tensors), max_len), pad_value, dtype=torch.long)
        for i, t in enumerate(tensors):
            out[i, :t.size(0)] = t
        return out

    return (
        pad(chosen_ids,       0),
        pad(chosen_labels,   -100),
        pad(rejected_ids,     0),
        pad(rejected_labels, -100),
    )

---

## 6. Generating a Toy DPO Dataset

For the nano model, we create a simple preference dataset where "chosen"
responses are more coherent Shakespeare excerpts and "rejected" responses
are shuffled/degraded versions:

In [ ]:
def make_toy_dpo_dataset(raw_text: str, output_path: str, n_samples: int = 300):
    """
    Toy DPO dataset for TinyShakespeare.
    Chosen = real Shakespeare excerpt.
    Rejected = same excerpt with word order randomly shuffled.
    """
    import random
    random.seed(42)

    words  = raw_text.split()
    chunks = [' '.join(words[i:i+60]) for i in range(0, len(words)-60, 60)]

    prompts = [
        "Write a short passage in the style of Shakespeare.",
        "Continue this dramatic scene.",
        "Write a soliloquy.",
        "Write dialogue for a play.",
    ]

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        for i in range(min(n_samples, len(chunks))):
            chosen   = chunks[i]
            # Rejected: shuffle words within the same excerpt
            words_chunk = chosen.split()
            random.shuffle(words_chunk)
            rejected = ' '.join(words_chunk)

            sample = {
                'prompt':   random.choice(prompts),
                'chosen':   chosen,
                'rejected': rejected,
            }
            f.write(json.dumps(sample) + '\n')

    print(f"Wrote {min(n_samples, len(chunks))} preference pairs to {output_path}")

---

## 7. The DPO Training Loop

[DPO requires two forward passes per batch: one through the policy model
(trained) and one through the reference model (frozen).]{.mark} The reference
model is a copy of the SFT model that never receives gradients.

Memory-efficient approach: run both forward passes sequentially, not
simultaneously. The reference model can be loaded in evaluation mode with
`torch.no_grad()` — its activations are never stored for backprop.

In [ ]:
import copy
import numpy as np
from pathlib import Path

def dpo_train(
    sft_model_path: str,
    data_path:      str,
    output_dir:     str,
    beta:           float = 0.1,
    max_lr:         float = 5e-5,   # very low — DPO is sensitive to LR
    min_lr:         float = 5e-6,
    warmup_steps:   int   = 50,
    max_steps:      int   = 500,
    batch_size:     int   = 2,      # small — DPO forward pass is expensive
    max_length:     int   = 256,
    eval_every:     int   = 100,
    use_lora:       bool  = True,   # highly recommended
    lora_rank:      int   = 8,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.bfloat16 if device.type == 'cuda' else torch.float32
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    from tutorial_02 import GPT, NanoGPTConfig
    from tutorial_03 import Tokenizer

    config = NanoGPTConfig()
    tok    = Tokenizer.load('nano_tokenizer.json')

    # ---- Policy model (trained) ----
    policy = GPT(config).to(device)
    ckpt   = torch.load(sft_model_path, map_location=device)
    policy.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)

    if use_lora:
        inject_lora(policy, rank=lora_rank)
        freeze_base_model(policy)
        trainable_params = [p for p in policy.parameters() if p.requires_grad]
    else:
        trainable_params = list(policy.parameters())

    # ---- Reference model (frozen copy of SFT model) ----
    ref_model = GPT(config).to(device)
    ref_model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

    print(f"Policy trainable params: {sum(p.numel() for p in trainable_params):,}")
    print(f"Reference model: frozen")

    # ---- Data ----
    ds = DPODataset(data_path, tok, max_length=max_length)
    val_size  = max(1, len(ds) // 10)
    train_ds, val_ds = torch.utils.data.random_split(ds, [len(ds) - val_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=collate_dpo, num_workers=1)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              collate_fn=collate_dpo, num_workers=1)

    # ---- Optimizer ----
    optimizer = torch.optim.AdamW(trainable_params, lr=max_lr, weight_decay=0.01)
    scheduler = make_cosine_schedule(optimizer, max_lr, min_lr,
                                      warmup_steps, max_steps)

    # ---- Training loop ----
    policy.train()
    train_iter     = iter(train_loader)
    history        = []

    for step in range(max_steps):
        try:
            chosen_ids, chosen_labels, rejected_ids, rejected_labels = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            chosen_ids, chosen_labels, rejected_ids, rejected_labels = next(train_iter)

        chosen_ids     = chosen_ids.to(device)
        chosen_labels  = chosen_labels.to(device)
        rejected_ids   = rejected_ids.to(device)
        rejected_labels = rejected_labels.to(device)

        with torch.autocast(device_type=device.type, dtype=dtype):
            # Policy log-probs
            policy_chosen_logps   = sequence_log_probs(policy, chosen_ids,
                                                        chosen_labels)
            policy_rejected_logps = sequence_log_probs(policy, rejected_ids,
                                                        rejected_labels)

            # Reference log-probs (no grad)
            with torch.no_grad():
                ref_chosen_logps   = sequence_log_probs(ref_model, chosen_ids,
                                                         chosen_labels)
                ref_rejected_logps = sequence_log_probs(ref_model, rejected_ids,
                                                         rejected_labels)

            loss, metrics = dpo_loss(
                policy_chosen_logps, policy_rejected_logps,
                ref_chosen_logps,    ref_rejected_logps,
                beta=beta,
            )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()
        scheduler.step()

        history.append(metrics)

        if step % 50 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(
                f"step {step:4d}  "
                f"loss={metrics['loss']:.4f}  "
                f"margin={metrics['reward_margin']:.4f}  "
                f"acc={metrics['accuracy']:.2%}  "
                f"lr={lr:.2e}"
            )

        # ---- Eval ----
        if step % eval_every == 0 and step > 0:
            policy.eval()
            eval_metrics = []
            with torch.no_grad():
                for c_ids, c_lab, r_ids, r_lab in val_loader:
                    c_ids  = c_ids.to(device);  c_lab  = c_lab.to(device)
                    r_ids  = r_ids.to(device);  r_lab  = r_lab.to(device)
                    with torch.autocast(device_type=device.type, dtype=dtype):
                        p_clog = sequence_log_probs(policy, c_ids, c_lab)
                        p_rlog = sequence_log_probs(policy, r_ids, r_lab)
                        rc_log = sequence_log_probs(ref_model, c_ids, c_lab)
                        rr_log = sequence_log_probs(ref_model, r_ids, r_lab)
                        _, m   = dpo_loss(p_clog, p_rlog, rc_log, rr_log, beta=beta)
                    eval_metrics.append(m)
            policy.train()

            avg = {k: np.mean([m[k] for m in eval_metrics]) for k in eval_metrics[0]}
            print(f"  [eval] loss={avg['loss']:.4f}  "
                  f"margin={avg['reward_margin']:.4f}  "
                  f"acc={avg['accuracy']:.2%}")

            torch.save({
                'step':    step,
                'policy':  policy.state_dict(),
                'metrics': avg,
                'beta':    beta,
            }, f'{output_dir}/dpo_step{step:04d}.pt')

    return policy, history

---

## 8. Monitoring DPO Training

DPO training has subtler failure modes than SFT. The loss alone is not
enough — you need to track all four diagnostic metrics:

### The reward margin

$$\text{margin} = \hat{r}(x, y_w) - \hat{r}(x, y_l)$$

**Healthy:** margin starts near 0 (policy = reference, same implicit reward
for both), increases steadily as training progresses — the policy is
learning to assign higher implicit reward to chosen responses.

**Pathological:** margin is negative (policy prefers rejected over chosen
— backward learning), or grows too fast then plateaus (policy has saturated
the preference signal — may need lower `β`).

### The accuracy

$$\text{acc} = P(\hat{r}(x, y_w) > \hat{r}(x, y_l))$$

Should increase from 0.5 (random) toward 0.8–0.9 over training. If it
plateaus at 0.5: the policy is not learning. If it reaches 1.0 too fast:
the model has memorized the preference data — reduce `β` or add more data.

### The chosen / rejected rewards individually

Watch `chosen_reward` and `rejected_reward` separately. Healthy DPO:
- `chosen_reward` increases (policy increasingly prefers chosen over ref)
- `rejected_reward` decreases (policy increasingly disprefers rejected vs ref)

[Pathological: both increase together — the policy is diverging from the
reference model uniformly]{.mark} (lowering `β` or the LR will fix this).

In [ ]:
def plot_dpo_training(history: list[dict]):
    import matplotlib.pyplot as plt

    steps = list(range(len(history)))
    fig, axes = plt.subplots(2, 2, figsize=(12, 7))
    fig.suptitle('DPO Training Diagnostics', fontsize=14, fontweight='bold')

    axes[0][0].plot(steps, [m['loss']            for m in history], color='#F44336')
    axes[0][0].set_title('DPO Loss');  axes[0][0].set_xlabel('Step')

    axes[0][1].plot(steps, [m['reward_margin']   for m in history], color='#4CAF50')
    axes[0][1].axhline(0, color='gray', linestyle='--', lw=0.8)
    axes[0][1].set_title('Reward Margin (chosen - rejected)'); axes[0][1].set_xlabel('Step')

    axes[1][0].plot(steps, [m['chosen_reward']   for m in history],
                    color='#2196F3', label='chosen')
    axes[1][0].plot(steps, [m['rejected_reward'] for m in history],
                    color='#FF9800', label='rejected')
    axes[1][0].axhline(0, color='gray', linestyle='--', lw=0.8)
    axes[1][0].set_title('Implicit Rewards'); axes[1][0].legend()
    axes[1][0].set_xlabel('Step')

    axes[1][1].plot(steps, [m['accuracy']        for m in history], color='#9C27B0')
    axes[1][1].axhline(0.5, color='gray', linestyle='--', lw=0.8, label='random')
    axes[1][1].set_ylim(0, 1); axes[1][1].set_title('Preference Accuracy')
    axes[1][1].legend(); axes[1][1].set_xlabel('Step')

    plt.tight_layout()
    plt.savefig('dpo_training.png', dpi=150)
    plt.show()

---

## Summary

| Concept | Key detail |
|---|---|
| RLHF objective | Maximize $\mathbb{E}[r(x,y)] - \beta \, D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}})$. |
| Optimal policy | $\pi^*(y\|x) \propto \pi_{\text{ref}}(y\|x) \exp(r/\beta)$. |
| DPO key step | Reward expressed in terms of optimal policy; $Z(x)$ cancels between chosen and rejected. |
| DPO loss | $-\log \sigma(\beta(\log\frac{\pi_\theta(y_w)}{\pi_\text{ref}(y_w)} - \log\frac{\pi_\theta(y_l)}{\pi_\text{ref}(y_l)}))$ |
| Reference model | Frozen copy of SFT model. Provides baseline log-probs. No gradients. |
| Implicit reward | $\hat{r}(x,y) = \beta \log(\pi_\theta(y\|x) / \pi_{\text{ref}}(y\|x))$. |
| $\beta$ | KL penalty strength. Typical: 0.1. Lower = more aggressive. Higher = more conservative. |
| Reward margin | $\hat{r}(y_w) - \hat{r}(y_l)$. Should be positive and growing. |
| Both rewards rising | Uniform divergence from ref — lower $\beta$ or LR. |
| Memory cost | [Two models in memory (policy + ref). With LoRA: policy is small; ref is full.]{.underline} |
| LR for DPO | Very low (5e-5). DPO is sensitive to LR — start low, increase only if margin doesn't grow. |

---

## Exercises

**1.** Derive the DPO loss from scratch. Starting from the KL-regularized
RLHF objective, show each algebraic step: the optimal policy, the
reward reparameterization, the Bradley-Terry substitution, and the
cancellation of $Z(x)$. Write out the derivation as a numbered series
of equations before looking at Section 2 again.

**2.** Implement a `verify_dpo_loss` function: construct a case where you
*know* the correct answer. Create a policy where $\pi_\theta = \pi_{\text{ref}}$
(i.e., zero LoRA updates, policy = reference). In this case, the chosen
and rejected log-ratios are both 0, the reward margin is 0, and the loss
should be $-\log \sigma(0) = \log 2 \approx 0.693$. Verify this numerically.

**3.** Run DPO with `β` ∈ {0.01, 0.1, 0.5} for 300 steps each. Plot the
four diagnostic metrics for each. Confirm that lower `β` leads to a
larger reward margin and higher accuracy but also a larger deviation
from the reference model (measure via mean absolute difference between
policy and reference log-probs on a held-out set).

**4.** Implement **identity preference data** — a sanity check where every
sample has `chosen = text[:half]` and `rejected = text[half:]`, split
from the same document. A model trained on this should learn nothing
useful (the preference is arbitrary) and accuracy should stay near 0.5
regardless of training duration. Verify this.

**5.** Implement the **length-normalized** variant of DPO: instead of summing
log-probs over the response, divide by the number of response tokens.
This prevents the model from preferring shorter responses (which naturally
have higher sum log-probs due to fewer terms). Compare the length
distributions of generated responses after standard DPO vs
length-normalized DPO on the toy dataset.

**6.** DPO requires the reference model to be stored in memory during
training. For large models, this is expensive. Implement
**reference-free DPO (SimPO)** as a drop-in alternative: the loss is
just $-\log \sigma(\beta(\text{avg\_log\_p}(y_w) - \text{avg\_log\_p}(y_l)))$,
where $\text{avg\_log\_p}$ is the mean (not sum) log-probability.
No reference model is needed. Compare SimPO vs DPO accuracy on the
toy dataset and note the tradeoff.